# 🧠 ResNet-50 Retinal Disease Detection
Optimized for RTX 4060 (BF16 + TF32)

In [ ]:
import os, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
print('Device:', device)

In [ ]:
class RFMiDDataset(Dataset):
    def __init__(self, img_dir, labels_df, transform=None):
        self.img_dir = img_dir
        self.labels_df = labels_df.reset_index(drop=True)
        self.transform = transform
        self.disease_cols = [c for c in labels_df.columns if c != 'ID']

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = self.labels_df.iloc[idx]['ID']
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        labels = self.labels_df.iloc[idx][self.disease_cols].values.astype(np.float32)
        return image, torch.tensor(labels)

In [ ]:
def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
    return transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

In [ ]:
DATA_DIR = 'data/rfmid'
train_df = pd.read_csv(os.path.join(DATA_DIR,'labels/train_labels.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR,'labels/val_labels.csv'))

train_dataset = RFMiDDataset(os.path.join(DATA_DIR,'images/train'), train_df, get_transforms(True))
val_dataset = RFMiDDataset(os.path.join(DATA_DIR,'images/val'), val_df, get_transforms(False))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
model = models.resnet50(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.disease_cols))
model = model.to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss/len(loader)

In [ ]:
for epoch in range(5):
    loss = train_epoch(model, train_loader)
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}')